In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from source.version5.model import EfficientModel
from source.version5.score import ScoreDataset, scoreModel

In [3]:
def score(fold):
    data = pd.read_csv('../../data/raw/data.csv')
    data = data[data['fold'] == fold].reset_index(drop=True)
    driver = data[['image_id','label']].copy()
    loader = {}
    loader['path'] = '../../data/raw/train_images/'
    loader['data'] = data
    valid = ScoreDataset(**loader)
    valid = DataLoader(valid, batch_size=10, shuffle=False, num_workers=1, drop_last=False)
    model = EfficientModel()
    weights = torch.load('../../model/version5/model_{}.pt'.format(fold), map_location='cpu')
    weights = weights['model_state_dict']
    weights = {name.replace('module.model','model'): param for name, param in weights.items()}
    try:
        weights.pop('n_averaged')
    except: 
        pass
    model.load_state_dict(weights)
    model = model.to('cuda:0')
    score = scoreModel(model, valid)
    score = pd.DataFrame(score)
    score.columns = ['score_0','score_1','score_2','score_3','score_4']
    driver = driver.join(score) 
    driver.to_csv('../../model/version5/valid_{}.csv'.format(fold), index=False)
    model.cpu()
    del model
    return None

In [4]:
score(0)

In [5]:
score(1)

In [6]:
score(2)

In [7]:
score(3)

In [8]:
score(4)

In [9]:
data0 = pd.read_csv('../../model/version5/valid_0.csv')
data1 = pd.read_csv('../../model/version5/valid_1.csv')
data2 = pd.read_csv('../../model/version5/valid_2.csv')
data3 = pd.read_csv('../../model/version5/valid_3.csv')
data4 = pd.read_csv('../../model/version5/valid_4.csv')

In [10]:
data = data0.append(data1).append(data2).append(data3).append(data4)
data = data.groupby(['image_id','label']).mean().reset_index()

In [11]:
data.to_csv('../../score/version5.csv', index=False)

In [12]:
data.shape

(21397, 7)

In [13]:
data.head()

,image_id,label,score_0,score_1,score_2,score_3,score_4
0,1000015157.jpg,0,0.538836,0.085718,0.248351,0.010003,0.117092
1,1000201771.jpg,3,0.000180,0.000367,0.003546,0.990958,0.004950
2,100042118.jpg,1,0.001607,0.006726,0.002059,0.044108,0.945500
3,1000723321.jpg,1,0.003487,0.883173,0.010696,0.077887,0.024757
4,1000812911.jpg,3,0.000205,0.000397,0.000291,0.998683,0.000423
